In [1]:
import pandas as pd
import torch
import numpy as np
from lightning.pytorch import Trainer
#from pytorch_lightning import Trainer
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer, Baseline
from pytorch_forecasting.data import NaNLabelEncoder
from pytorch_forecasting.metrics import SMAPE
from pytorch_lightning.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("/Users/nik/github/ridership-prediction/model/data/cleaned_data.csv")

In [3]:
df.head()

,date,time,origin,destination,ridership,day_of_week,is_weekend,is_holiday
0,2025-01-01,00:00,Abdullah Hukum,Klang,1,3,0,1
1,2025-01-01,00:00,Abdullah Hukum,Telok Pulai,1,3,0,1
2,2025-01-01,00:00,Bangi,Batu Caves,1,3,0,1
3,2025-01-01,00:00,Bank Negara,Sungai Gadut,1,3,0,1
4,2025-01-01,00:00,Batu Tiga,Kampung Raja Uda,1,3,0,1


In [4]:
df.columns

Index(['date', 'time', 'origin', 'destination', 'ridership', 'day_of_week',
       'is_weekend', 'is_holiday'],
      dtype='object')

In [5]:
df.dtypes

date           object
time           object
origin         object
destination    object
ridership       int64
day_of_week     int64
is_weekend      int64
is_holiday      int64
dtype: object

In [6]:
# pre-process by combine datetime, change datetime64 dtypes 
df['date'] = df['date'].astype(str)
df['time'] = df['time'].astype(str)
df['timestamp'] = df['date']+' '+df['time']
df['timestamp'] = pd.to_datetime(df['timestamp'], format='%Y-%m-%d %H:%M')

In [7]:
df['route'] = df['origin']+'_to_'+df['destination']

In [8]:
# sort by route, then time
df = df.sort_values(by=['route', 'timestamp']).reset_index(drop=True)

In [9]:
# create new time_idx, the time series
df['time_idx'] = df.groupby('route').cumcount()
#covariate for dayofweek
df['dayofweek_sin'] = np.sin(2 * np.pi * (df['day_of_week'] - 1) / 7)
df['dayofweek_cos'] = np.cos(2 * np.pi * (df['day_of_week'] - 1) / 7)
#covariate holiday and weekend is hot code, taken care of
print(df.head(20))

          date   time          origin     destination  ridership  day_of_week  \
0   2025-01-03  22:00  Abdullah Hukum  Abdullah Hukum          2            5   
1   2025-01-05  14:00  Abdullah Hukum  Abdullah Hukum          1            7   
2   2025-01-06  19:00  Abdullah Hukum  Abdullah Hukum          2            1   
3   2025-01-07  16:00  Abdullah Hukum  Abdullah Hukum          1            2   
4   2025-01-07  19:00  Abdullah Hukum  Abdullah Hukum          1            2   
5   2025-01-07  20:00  Abdullah Hukum  Abdullah Hukum          7            2   
6   2025-01-08  06:00  Abdullah Hukum  Abdullah Hukum          1            3   
7   2025-01-08  21:00  Abdullah Hukum  Abdullah Hukum          2            3   
8   2025-01-09  03:00  Abdullah Hukum  Abdullah Hukum          1            4   
9   2025-01-09  11:00  Abdullah Hukum  Abdullah Hukum          1            4   
10  2025-01-09  22:00  Abdullah Hukum  Abdullah Hukum          1            4   
11  2025-01-11  00:00  Abdul

In [10]:
group_counts = df.groupby("route").size()
valid_groups = group_counts[group_counts > 40].index
df = df[df["route"].isin(valid_groups)]

routes = df['route'].unique()
train_routes, val_routes = train_test_split(routes, test_size=0.2, random_state=42)

train_df = df[df['route'].isin(train_routes)]
val_df = df[df['route'].isin(val_routes)]

In [11]:
max_encoder_length = 30  # lookback
max_prediction_length = 7  # forecast horizon

training = TimeSeriesDataSet(
    train_df,
    time_idx="time_idx",
    target="ridership",
    group_ids=["route"],
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,
    static_categoricals=["route"],
    time_varying_known_reals=["time_idx", "dayofweek_sin", "dayofweek_cos", "is_weekend", "is_holiday"],
    time_varying_unknown_reals=["ridership"],
    target_normalizer=NaNLabelEncoder(),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
)

In [12]:
#print total series loaded
total_series = df["route"].nunique()
print(f"Total series loaded: {total_series}")

Total series loaded: 1733


In [13]:
train_dataloader = training.to_dataloader(train=True, batch_size=64, num_workers=1)
train_dataloader = training.to_dataloader(train=True, batch_size=64, num_workers=1)
val_dataloader = validation.to_dataloader(train=False, batch_size=64, num_workers=1)

NameError: name 'validation' is not defined

In [ ]:
early_stop_callback = EarlyStopping(monitor="val_loss", patience=5, verbose=True, mode="min")

trainer = Trainer(
    max_epochs=30,
    gradient_clip_val=0.1,
    callbacks=[early_stop_callback],
)

In [ ]:
tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.02,
    hidden_size=32,
    attention_head_size=1,
    dropout=0.1,
    loss=SMAPE(),
    log_interval=10,
)

trainer.fit(
    tft,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader,
)
predictions = tft.predict(val_dataloader)
